# 02 · Silver — clean and conform trips
Reads the Bronze table **`lh_bronze.dbo.raw_data`** (a OneLake table shortcut that transforms
`Files/raw_data/*.xlsx` into Delta automatically) and writes **`lh_silver.dbo.trips`**
(one validated row per booking) plus **`lh_silver.dbo.trips_rejected`** (quarantined rows with the reason).

Data-quality rules
| Rule | Action |
|---|---|
| `"NULL"`, `"NA"`, empty strings | → null |
| `trip_end_date` arrives as text (`M/d/yy H:mm`) | parsed; Excel zero date `1899-12-30` → null |
| `trip_start_date`, `actual_eta`, `Planned_ETA`, `Data_Ping_time` are empty after the shortcut's type inference | dropped; **booking date** is the trip date |
| Missing `BookingID` or booking date | quarantined |
| Duplicate `BookingID` | keep one row |
| Trip end before booking date | lead time → null, flagged |
| `Driver_MobileNo` is personal data | dropped |
| Location strings `Place, City, State` | parsed; PIN codes stripped; missing state inferred from the city |
| Vehicle type variants (`24 / 26` vs `24 | 26`), plate prefixes (`.TN…`) | normalised |
| `ontime = G` → On Time, `delay = R` → Delayed (wins if both), neither → Unknown | `delivery_status` |

In [ ]:
bronze_table = "lh_bronze.dbo.raw_data"
silver_table = "dbo.trips"
rejected_table = "dbo.trips_rejected"

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

src = spark.table(bronze_table)
print(f"Bronze rows: {src.count():,}")

## Cleansing helpers

In [ ]:
NULL_TOKENS = ["", "NULL", "NA", "N/A", "NAN", "NONE", "-"]


def clean(col_name):
    c = F.trim(F.regexp_replace(F.col(col_name), r"\s+", " "))
    return F.when(F.upper(c).isin(NULL_TOKENS), None).otherwise(c)


def to_ts(col_name, fmt="M/d/yy H:mm"):
    """Parse 'M/d/yy H:mm' text; anything else (e.g. the Excel zero date 1899-12-30) -> null."""
    c = clean(col_name)
    return F.when(c.rlike(r"^\d{1,2}/\d{1,2}/\d{2} \d{1,2}:\d{2}$"), F.to_timestamp(c, fmt))


def to_num(col_name):
    return clean(col_name).cast("double")


def lat_lon(col_name, idx):
    return F.split(clean(col_name), ",").getItem(idx).cast("double")


STATE_FIXES = {"Pondicherry": "Puducherry", "Chattisgarh": "Chhattisgarh", "Orissa": "Odisha"}
INDIAN_STATES = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh", "Goa", "Gujarat", "Haryana",
    "Himachal Pradesh", "Jharkhand", "Karnataka", "Kerala", "Madhya Pradesh", "Maharashtra", "Manipur",
    "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Punjab", "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana",
    "Tripura", "Uttar Pradesh", "Uttarakhand", "West Bengal", "Delhi", "Puducherry", "Chandigarh",
    "Jammu & Kashmir", "Dadra & Nagar Haveli", "Daman & Diu", "Ladakh", "Lakshadweep",
]


def fix_state(state):
    for wrong, right in STATE_FIXES.items():
        state = F.when(state == wrong, F.lit(right)).otherwise(state)
    return state


def location_parts(col_name):
    """'Place, City, State[, India]' -> (place, city, state). PIN codes are stripped; a lone
    state name (e.g. 'Jharkhand, India') becomes the state with no city."""
    tokens = F.filter(
        F.transform(F.split(clean(col_name), ","), lambda t: F.trim(F.regexp_replace(t, r"\b\d{6}\b", ""))),
        lambda t: (t != "") & (F.upper(t) != "INDIA"))
    n = F.size(tokens)
    first = fix_state(F.initcap(F.element_at(tokens, 1)))
    lone_state = (n == 1) & first.isin(INDIAN_STATES)
    state = F.when(n >= 2, fix_state(F.initcap(F.element_at(tokens, -1)))).when(lone_state, first)
    city = F.initcap(F.when(n >= 3, F.element_at(tokens, -2)).when(~lone_state, F.element_at(tokens, 1)))
    place = (F.when(n >= 3, F.array_join(F.slice(tokens, F.lit(1), n - 2), ", "))
              .when(~lone_state, F.element_at(tokens, 1)))
    return place, city, state

## Conform

In [ ]:
o_place, o_city, o_state = location_parts("Origin_Location")
d_place, d_city, d_state = location_parts("Destination_Location")
ontime = F.upper(clean("ontime")) == "G"
delayed = F.upper(clean("delay")) == "R"

conformed = src.select(
    clean("BookingID").alias("booking_id"),
    F.initcap(clean("Market/Regular ")).alias("booking_type"),
    F.to_date("BookingID_Date").alias("booking_date"),
    F.upper(F.regexp_replace(clean("vehicle_no"), r"[^0-9A-Za-z]", "")).alias("vehicle_no"),
    F.regexp_replace(F.regexp_replace(clean("vehicleType"), r"\s*\|\s*", " / "), r"\s*-\s*$", "").alias("vehicle_type"),
    F.upper(clean("GpsProvider")).alias("gps_provider"),
    clean("Origin_Location").alias("origin_location"),
    clean("OriginLocation_Code").alias("origin_location_code"),
    o_place.alias("origin_place"), o_city.alias("origin_city"), o_state.alias("origin_state"),
    lat_lon("Org_lat_lon", 0).alias("origin_lat"), lat_lon("Org_lat_lon", 1).alias("origin_lon"),
    clean("Destination_Location").alias("destination_location"),
    clean("DestinationLocation_Code").alias("destination_location_code"),
    d_place.alias("destination_place"), d_city.alias("destination_city"), d_state.alias("destination_state"),
    lat_lon("Des_lat_lon", 0).alias("destination_lat"), lat_lon("Des_lat_lon", 1).alias("destination_lon"),
    clean("Current_Location").alias("last_gps_location"),
    to_num("Curr_lat").alias("last_gps_lat"), to_num("Curr_lon").alias("last_gps_lon"),
    to_ts("trip_end_date").alias("trip_end_ts"),
    to_num("TRANSPORTATION_DISTANCE_IN_KM").alias("distance_km"),
    to_num("Minimum_kms_to_be_covered_in_a_day").alias("min_km_per_day"),
    F.initcap(clean("Driver_Name")).alias("driver_name"),
    F.upper(clean("customerID")).alias("customer_id"),
    clean("customerNameCode").alias("customer_name"),
    F.upper(clean("supplierID")).alias("supplier_id"),
    clean("supplierNameCode").alias("supplier_name"),
    F.upper(clean("Material Shipped")).alias("material_shipped"),
    F.when(delayed, "Delayed").when(ontime, "On Time").otherwise("Unknown").alias("delivery_status"),
    F.col("__filepath__").alias("_source_file"),
    F.col("__ingestion_timestamp__").alias("_source_ingested_at"),
)

# Fill a missing state from the most common state recorded for the same city elsewhere in the data.
city_state = (
    conformed.select(F.col("origin_city").alias("city"), F.col("origin_state").alias("state"))
    .unionByName(conformed.select(F.col("destination_city").alias("city"), F.col("destination_state").alias("state")))
    .filter(F.col("city").isNotNull() & F.col("state").isNotNull())
    .groupBy("city", "state").count()
    .withColumn("_rn", F.row_number().over(Window.partitionBy("city").orderBy(F.desc("count"), "state")))
    .filter("_rn = 1").select("city", F.col("state").alias("_inferred_state"))
)
for role in ("origin", "destination"):
    conformed = (conformed
                 .join(F.broadcast(city_state.withColumnRenamed("city", f"{role}_city")), f"{role}_city", "left")
                 .withColumn(f"{role}_state", F.coalesce(f"{role}_state", "_inferred_state"))
                 .drop("_inferred_state"))

booking_ts = F.col("booking_date").cast("timestamp")
conformed = (
    conformed
    .withColumn("dq_end_before_booking", F.coalesce(F.col("trip_end_ts") < booking_ts, F.lit(False)))
    .withColumn("dq_missing_distance", F.col("distance_km").isNull())
    .withColumn("lead_time_days",  # booking date -> trip end
                F.when(~F.col("dq_end_before_booking"),
                       F.round((F.unix_timestamp("trip_end_ts") - F.unix_timestamp(booking_ts)) / 86400.0, 2)))
    .withColumn("is_completed", F.col("trip_end_ts").isNotNull())
    .withColumn("_processed_at", F.current_timestamp())
)

## Quarantine, de-duplicate, write

In [ ]:
reject_reason = (F.when(F.col("booking_id").isNull(), "missing booking_id")
                  .when(F.col("booking_date").isNull(), "missing booking date"))
rejected = conformed.withColumn("reject_reason", reject_reason).filter(F.col("reject_reason").isNotNull())
valid = conformed.filter(reject_reason.isNull())

w = Window.partitionBy("booking_id").orderBy(F.col("trip_end_ts").desc_nulls_last())
silver = valid.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

(silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table))
(rejected.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(rejected_table))
spark.sql(f"OPTIMIZE {silver_table}")

## Validate

In [ ]:
s = spark.table(silver_table)
total, unique_keys = s.count(), s.select("booking_id").distinct().count()
n_rejected = spark.table(rejected_table).count()
n_in = src.count()
print(f"Bronze rows: {n_in:,} | silver: {total:,} | rejected: {n_rejected:,} | duplicates removed: {n_in - total - n_rejected:,}")
assert total == unique_keys, "booking_id is not unique in Silver"
assert total > 0.95 * n_in, "More than 5% of rows were dropped - check Bronze data"
display(s.groupBy("delivery_status").count().orderBy("delivery_status"))
display(s.select(*[F.sum(F.col(c).cast("int")).alias(c) for c in
                   ["dq_end_before_booking", "dq_missing_distance", "is_completed"]]))